In [ ]:
##  %matplotlib inline
%matplotlib widget

import matplotlib as mpl
mpl.rcParams['animation.html'] = 'jshtml'
mpl.rcParams['animation.embed_limit'] = 200.0

import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import happi

In [ ]:
path = "../work_dir/tuto_3d"
# path = "../work_dir/tuto_3d_PML3b"
# path = "../work_dir/tuto_3d_PML3b"
# path = "../work_dir/tuto_3d_PML"
path = "../work_dir/rot_anal/t2"
path = "../work_dir/rot_anal/t-Gauss"
path = "../work_dir/benchmark00"
field_component = "Bz"
axes_aspect = 'auto' # 'equal'


S = happi.Open(path, verbose=False)
print(S.namelist.Main.timestep) 
print(S.namelist.Main.geometry)
for species in S.namelist.Species:
    print("species "+species.name+" has mass "+str(species.mass))

Diag = S.Field(0,"Ex")
xxx = Diag.getData()
print(np.shape(xxx))

In [ ]:
%matplotlib widget

import happi
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.widgets import Slider

# =========================================================
# USER OPTIONS
# =========================================================

FIELD = "Bz"  # "Ex", "Ey", "By", ...

# Options:
#   "adaptive"       -> each subplot uses its own current abs(max)
#   "global_plotted" -> all displayed subplots share max abs value
COLORBAR_MODE = "global_plotted"

cmap = "RdBu_r"

AXES = ["x", "y", "z"]

# Fallback if the plotted data are exactly zero
VMAX_FALLBACK = 1.0

# =========================================================
# OPEN SIMULATION
# =========================================================

S = happi.Open(path, verbose=False)

# =========================================================
# GEOMETRY
# =========================================================

ax_idx = {"x": 0, "y": 1, "z": 2}

N_spatial = {}
spatial_values = {}

for AXIS in AXES:

    L = S.namelist.Main.grid_length[ax_idx[AXIS]]
    d = S.namelist.Main.cell_length[ax_idx[AXIS]]

    N = int(round(L / d))

    N_spatial[AXIS] = N
    spatial_values[AXIS] = np.arange(N) * d

# =========================================================
# TIMESTEPS
# =========================================================

F0 = S.Field(0, FIELD)
timesteps = F0.getTimesteps()

# Optional reduction
# timesteps = timesteps[::10]

# =========================================================
# INITIAL INDICES
# =========================================================

itime = 0

i_spatial = {
    "x": N_spatial["x"] // 2,
    "y": N_spatial["y"] // 2,
    "z": N_spatial["z"] // 2,
}

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def load_slice(AXIS, t, idx):
    """
    Load one 2D slice normal to AXIS at timestep t and spatial index idx.
    """
    coord = spatial_values[AXIS][idx]

    D = S.Field(
        0,
        FIELD,
        timesteps=t,
        subset={AXIS: [coord]},
    )

    data = D.getData()[0]

    return data, coord


def safe_absmax(arrays):
    """
    Return max(abs(data)) over one or more arrays.
    Keeps colour limits finite and non-zero.
    """
    vmax = 0.0

    for arr in arrays:
        local_max = np.nanmax(np.abs(arr))
        vmax = max(vmax, float(local_max))

    if not np.isfinite(vmax) or vmax == 0.0:
        vmax = VMAX_FALLBACK

    return vmax


def get_vmax_by_axis(data_by_axis):
    """
    Compute symmetric colour limits according to COLORBAR_MODE.
    """
    if COLORBAR_MODE == "adaptive":
        return {
            AXIS: safe_absmax([data_by_axis[AXIS]])
            for AXIS in AXES
        }

    elif COLORBAR_MODE == "global_plotted":
        global_vmax = safe_absmax(data_by_axis.values())
        return {
            AXIS: global_vmax
            for AXIS in AXES
        }

    else:
        raise ValueError(
            "Unknown COLORBAR_MODE. Use 'adaptive' or 'global_plotted'."
        )

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

plt.subplots_adjust(bottom=0.22, wspace=0.30)

ims = {}
titles = {}
cbars = {}

# =========================================================
# INITIAL DATA
# =========================================================

t0 = timesteps[itime]

data_by_axis = {}
coord_by_axis = {}

for AXIS in AXES:
    data, coord = load_slice(AXIS, t0, i_spatial[AXIS])
    data_by_axis[AXIS] = data
    coord_by_axis[AXIS] = coord

vmax_by_axis = get_vmax_by_axis(data_by_axis)

# =========================================================
# INITIAL PLOTS
# =========================================================

for ax, AXIS in zip(axes, AXES):

    data = data_by_axis[AXIS]
    coord = coord_by_axis[AXIS]

    vmax = vmax_by_axis[AXIS]
    vmin = -vmax

    im = ax.imshow(
        data.T,
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )

    ims[AXIS] = im

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(FIELD)
    cbars[AXIS] = cbar

    titles[AXIS] = ax.set_title(
        f"{AXIS} slice | t={t0} | "
        f"{AXIS}={coord:.2f}"
    )

    label_names = [foo for foo in ax_idx.keys() if foo != AXIS]

    ax.set_xlabel(label_names[0])
    ax.set_ylabel(label_names[1])

# =========================================================
# SLIDERS
# =========================================================

# Shared time slider
ax_time = plt.axes([0.15, 0.12, 0.70, 0.03])

slider_time = Slider(
    ax_time,
    "time index",
    0,
    len(timesteps) - 1,
    valinit=itime,
    valstep=1,
)

# Individual spatial sliders
slider_axes = {}
slider_spatial = {}

slider_positions = {
    "x": [0.10, 0.05, 0.22, 0.03],
    "y": [0.39, 0.05, 0.22, 0.03],
    "z": [0.68, 0.05, 0.22, 0.03],
}

for AXIS in AXES:

    ax_slider = plt.axes(slider_positions[AXIS])

    slider = Slider(
        ax_slider,
        f"{AXIS} index",
        0,
        N_spatial[AXIS] - 1,
        valinit=i_spatial[AXIS],
        valstep=1,
    )

    slider_axes[AXIS] = ax_slider
    slider_spatial[AXIS] = slider

# =========================================================
# UPDATE FUNCTION
# =========================================================

def update(val):

    it = int(slider_time.val)
    t = timesteps[it]

    # ---------------------------------------------
    # First load all plotted data
    # ---------------------------------------------

    data_by_axis = {}
    coord_by_axis = {}

    for AXIS in AXES:

        idx = int(slider_spatial[AXIS].val)

        data, coord = load_slice(AXIS, t, idx)

        data_by_axis[AXIS] = data
        coord_by_axis[AXIS] = coord

    # ---------------------------------------------
    # Then compute colour limits
    # ---------------------------------------------

    vmax_by_axis = get_vmax_by_axis(data_by_axis)

    # ---------------------------------------------
    # Update each subplot
    # ---------------------------------------------

    for AXIS in AXES:

        new_data = data_by_axis[AXIS]
        coord = coord_by_axis[AXIS]

        vmax = vmax_by_axis[AXIS]
        vmin = -vmax

        ims[AXIS].set_data(new_data.T)
        ims[AXIS].set_clim(vmin, vmax)

        cbars[AXIS].update_normal(ims[AXIS])

        titles[AXIS].set_text(
            f"{AXIS} slice | t={t} | "
            f"{AXIS}={coord:.2f}"
        )

    fig.canvas.draw_idle()

# =========================================================
# CONNECT
# =========================================================

slider_time.on_changed(update)

for AXIS in AXES:
    slider_spatial[AXIS].on_changed(update)

plt.show()

In [ ]:
# Recommended for this redraw-on-slider-change procedure
%matplotlib inline

import happi
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    IntSlider,
    SelectionSlider,
    VBox,
    HBox,
    interactive_output,
)
from IPython.display import display

# =========================================================
# USER OPTIONS
# =========================================================

# Manual switch: choose field family before running the cell
FIELD_FAMILY = "B"        # "B" or "E"

# Manual switch: choose colourbar behaviour before running the cell
COLORBAR_MODE = "global"  # "global" or "adaptive"

cmap = "RdBu_r"

AXES = ["x", "y", "z"]

# If > 1, both plotting and global scan use only every N-th timestep.
# Keep this as 1 if you want the true global maximum over all saved timesteps.
TIMESTEP_STRIDE = 1

# Fallback only used if the selected data are exactly zero or invalid
VMAX_FALLBACK = 1.0

# =========================================================
# OPEN SIMULATION
# =========================================================

S = happi.Open(path, verbose=False)

# =========================================================
# GEOMETRY
# =========================================================

ax_idx = {
    "x": 0,
    "y": 1,
    "z": 2,
}

N_spatial = {}
spatial_values = {}

for AXIS in AXES:

    L = S.namelist.Main.grid_length[ax_idx[AXIS]]
    d = S.namelist.Main.cell_length[ax_idx[AXIS]]

    N = int(round(L / d))

    N_spatial[AXIS] = N
    spatial_values[AXIS] = np.arange(N) * d

# =========================================================
# TIMESTEPS
# =========================================================

# Use one representative field only to get the available timesteps
if FIELD_FAMILY.upper() == "B":
    REFERENCE_FIELD = "By"
elif FIELD_FAMILY.upper() == "E":
    REFERENCE_FIELD = "Ey"
else:
    raise ValueError("FIELD_FAMILY must be either 'B' or 'E'.")

F0 = S.Field(0, REFERENCE_FIELD)
timesteps = np.asarray(F0.getTimesteps())

timesteps = timesteps[::TIMESTEP_STRIDE]

if len(timesteps) == 0:
    raise RuntimeError("No timesteps available after applying TIMESTEP_STRIDE.")

# =========================================================
# POLARISATION COMPONENTS
# =========================================================

FIELD_FAMILY = FIELD_FAMILY.upper()
COLORBAR_MODE = COLORBAR_MODE.lower()

if FIELD_FAMILY == "B":
    POLARISATIONS = ["Bx", "By", "Bz"]
elif FIELD_FAMILY == "E":
    POLARISATIONS = ["Ex", "Ey", "Ez"]
else:
    raise ValueError("FIELD_FAMILY must be either 'B' or 'E'.")

if COLORBAR_MODE not in ["global", "adaptive"]:
    raise ValueError("COLORBAR_MODE must be either 'global' or 'adaptive'.")

# Start from y-component by default, if available
DEFAULT_POLARISATION = POLARISATIONS[1]

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def raw_absmax(data):
    """
    Return max(abs(data)) without imposing a fallback.
    """
    arr = np.asarray(data)

    if arr.size == 0:
        return 0.0

    abs_arr = np.abs(arr)

    if np.all(np.isnan(abs_arr)):
        return 0.0

    val = np.nanmax(abs_arr)

    if not np.isfinite(val):
        return 0.0

    return float(val)


def finalise_vmax(vmax):
    """
    Ensure the colour scale is finite and non-zero.
    """
    if not np.isfinite(vmax) or vmax <= 0.0:
        return VMAX_FALLBACK

    return float(vmax)


def load_plane(FIELD, t, normal_axis, normal_index):
    """
    Load a 2D plane by fixing one spatial coordinate.

    normal_axis:
        "z" -> xy plane
        "y" -> xz plane
        "x" -> yz plane
    """
    coord = spatial_values[normal_axis][normal_index]

    D = S.Field(
        0,
        FIELD,
        timesteps=t,
        subset={normal_axis: [coord]},
    )

    data = np.asarray(D.getData()[0])
    data = np.squeeze(data)

    return data, coord


def plane_extent(horizontal_axis, vertical_axis):
    """
    Extent for imshow.
    """
    h = spatial_values[horizontal_axis]
    v = spatial_values[vertical_axis]

    return [
        h[0],
        h[-1],
        v[0],
        v[-1],
    ]


def limits_from_data(data):
    """
    Adaptive symmetric colour limits for one subplot.
    """
    vmax = finalise_vmax(raw_absmax(data))
    return -vmax, vmax


def global_limits():
    """
    Fixed symmetric colour limits for global mode.
    """
    return -GLOBAL_VMAX, GLOBAL_VMAX

# =========================================================
# GLOBAL MAXIMUM SCAN
# =========================================================

GLOBAL_VMAX = None

if COLORBAR_MODE == "global":

    print("Computing global normalisation...")
    print(f"Field family: {FIELD_FAMILY}")
    print(f"Components: {POLARISATIONS}")
    print(f"Number of timesteps used: {len(timesteps)}")

    vmax_scan = 0.0
    counter = 0
    total = len(POLARISATIONS) * len(timesteps)

    for FIELD in POLARISATIONS:

        for t in timesteps:

            counter += 1

            D = S.Field(
                0,
                FIELD,
                timesteps=t,
            )

            data = D.getData()[0]
            vmax_scan = max(vmax_scan, raw_absmax(data))

            print(
                f"\rScanning {counter}/{total}: {FIELD}, t={t}",
                end="",
            )

    GLOBAL_VMAX = finalise_vmax(vmax_scan)

    print()
    print(f"GLOBAL_VMAX = {GLOBAL_VMAX}")

else:
    print("Adaptive colourbar mode selected.")
    print("No global scan will be performed.")

# =========================================================
# PLOTTING FUNCTION
# =========================================================

def plot_three_planes(time_index, FIELD, ix, iy, iz):

    t = timesteps[time_index]

    # -----------------------------------------------------
    # Load current slices
    # -----------------------------------------------------

    # xy plane: z fixed
    field_xy, z_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="z",
        normal_index=iz,
    )

    # xz plane: y fixed
    field_xz, y_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="y",
        normal_index=iy,
    )

    # yz plane: x fixed
    field_yz, x_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="x",
        normal_index=ix,
    )

    # -----------------------------------------------------
    # Colour limits
    # -----------------------------------------------------

    if COLORBAR_MODE == "global":

        vmin_xy, vmax_xy = global_limits()
        vmin_xz, vmax_xz = global_limits()
        vmin_yz, vmax_yz = global_limits()

    elif COLORBAR_MODE == "adaptive":

        vmin_xy, vmax_xy = limits_from_data(field_xy)
        vmin_xz, vmax_xz = limits_from_data(field_xz)
        vmin_yz, vmax_yz = limits_from_data(field_yz)

    else:
        raise ValueError("Unknown COLORBAR_MODE.")

    # -----------------------------------------------------
    # Figure
    # -----------------------------------------------------

    fig, axs = plt.subplots(
        1,
        3,
        figsize=(18, 6),
        constrained_layout=True,
    )

    fig.suptitle(
        f"{FIELD} | t index = {time_index}, t = {t} | "
        f"colourbar mode = {COLORBAR_MODE}",
        fontsize=14,
    )

    # -----------------------------------------------------
    # xy plane, z fixed
    # -----------------------------------------------------

    im0 = axs[0].imshow(
        field_xy.T,
        extent=plane_extent("x", "y"),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin_xy,
        vmax=vmax_xy,
    )

    axs[0].set_title(
        f"xy plane | z = {z_coord:.4g}"
    )
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")

    cbar0 = fig.colorbar(
        im0,
        ax=axs[0],
        orientation="horizontal",
        pad=0.12,
    )
    cbar0.set_label(FIELD)

    # -----------------------------------------------------
    # xz plane, y fixed
    # -----------------------------------------------------

    im1 = axs[1].imshow(
        field_xz.T,
        extent=plane_extent("x", "z"),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin_xz,
        vmax=vmax_xz,
    )

    axs[1].set_title(
        f"xz plane | y = {y_coord:.4g}"
    )
    axs[1].set_xlabel("x")
    axs[1].set_ylabel("z")

    cbar1 = fig.colorbar(
        im1,
        ax=axs[1],
        orientation="horizontal",
        pad=0.12,
    )
    cbar1.set_label(FIELD)

    # -----------------------------------------------------
    # yz plane, x fixed
    # -----------------------------------------------------

    im2 = axs[2].imshow(
        field_yz.T,
        extent=plane_extent("y", "z"),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=vmin_yz,
        vmax=vmax_yz,
    )

    axs[2].set_title(
        f"yz plane | x = {x_coord:.4g}"
    )
    axs[2].set_xlabel("y")
    axs[2].set_ylabel("z")

    cbar2 = fig.colorbar(
        im2,
        ax=axs[2],
        orientation="horizontal",
        pad=0.12,
    )
    cbar2.set_label(FIELD)

    plt.show()
    plt.close(fig)

# =========================================================
# SLIDERS
# =========================================================

itime0 = 0

ix0 = N_spatial["x"] // 2
iy0 = N_spatial["y"] // 2
iz0 = N_spatial["z"] // 2

slider_time = IntSlider(
    min=0,
    max=len(timesteps) - 1,
    step=1,
    value=itime0,
    description="time index",
    continuous_update=False,
)

slider_pol = SelectionSlider(
    options=POLARISATIONS,
    value=DEFAULT_POLARISATION,
    description="polarisation",
    continuous_update=False,
)

slider_x = IntSlider(
    min=0,
    max=N_spatial["x"] - 1,
    step=1,
    value=ix0,
    description="x index",
    continuous_update=False,
)

slider_y = IntSlider(
    min=0,
    max=N_spatial["y"] - 1,
    step=1,
    value=iy0,
    description="y index",
    continuous_update=False,
)

slider_z = IntSlider(
    min=0,
    max=N_spatial["z"] - 1,
    step=1,
    value=iz0,
    description="z index",
    continuous_update=False,
)

# =========================================================
# INTERACTIVE OUTPUT
# =========================================================

out = interactive_output(
    plot_three_planes,
    {
        "time_index": slider_time,
        "FIELD": slider_pol,
        "ix": slider_x,
        "iy": slider_y,
        "iz": slider_z,
    },
)

display(
    VBox(
        [
            HBox([slider_time, slider_pol]),
            HBox([slider_x, slider_y, slider_z]),
            out,
        ]
    )
)

In [ ]:
%matplotlib widget

import happi
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import IntSlider, SelectionSlider, VBox, HBox
from IPython.display import display

# =========================================================
# USER OPTIONS
# =========================================================

FIELD_FAMILY = "B"        # "B" or "E"
COLORBAR_MODE = "adaptive"  # "global" or "adaptive"

cmap = "RdBu_r"

AXES = ["x", "y", "z"]

# Use 1 for a true scan over all selected timesteps.
# Increase only if the global scan is too expensive.
TIMESTEP_STRIDE = 1

VMAX_FALLBACK = 1.0

# =========================================================
# OPEN SIMULATION
# =========================================================

S = happi.Open(path, verbose=False)

# =========================================================
# FIELD FAMILY / POLARISATIONS
# =========================================================

FIELD_FAMILY = FIELD_FAMILY.upper()
COLORBAR_MODE = COLORBAR_MODE.lower()

if FIELD_FAMILY == "B":
    POLARISATIONS = ["Bx", "By", "Bz"]
    REFERENCE_FIELD = "By"
elif FIELD_FAMILY == "E":
    POLARISATIONS = ["Ex", "Ey", "Ez"]
    REFERENCE_FIELD = "Ey"
else:
    raise ValueError("FIELD_FAMILY must be either 'B' or 'E'.")

if COLORBAR_MODE not in ["global", "adaptive"]:
    raise ValueError("COLORBAR_MODE must be either 'global' or 'adaptive'.")

DEFAULT_FIELD = POLARISATIONS[1]

# =========================================================
# GEOMETRY
# =========================================================

ax_idx = {
    "x": 0,
    "y": 1,
    "z": 2,
}

N_spatial = {}
spatial_values = {}

for AXIS in AXES:

    L = S.namelist.Main.grid_length[ax_idx[AXIS]]
    d = S.namelist.Main.cell_length[ax_idx[AXIS]]

    N = int(round(L / d))

    N_spatial[AXIS] = N
    spatial_values[AXIS] = np.arange(N) * d

# =========================================================
# TIMESTEPS
# =========================================================

F0 = S.Field(0, REFERENCE_FIELD)
timesteps = np.asarray(F0.getTimesteps())
timesteps = timesteps[::TIMESTEP_STRIDE]

if len(timesteps) == 0:
    raise RuntimeError("No timesteps available after applying TIMESTEP_STRIDE.")

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def raw_absmax(data):
    arr = np.asarray(data)

    if arr.size == 0:
        return 0.0

    abs_arr = np.abs(arr)

    if np.all(np.isnan(abs_arr)):
        return 0.0

    val = np.nanmax(abs_arr)

    if not np.isfinite(val):
        return 0.0

    return float(val)


def finalise_vmax(vmax):
    if not np.isfinite(vmax) or vmax <= 0.0:
        return VMAX_FALLBACK

    return float(vmax)


def load_plane(FIELD, t, normal_axis, normal_index):
    """
    normal_axis = "z" gives xy plane
    normal_axis = "y" gives xz plane
    normal_axis = "x" gives yz plane
    """
    coord = spatial_values[normal_axis][normal_index]

    D = S.Field(
        0,
        FIELD,
        timesteps=t,
        subset={normal_axis: [coord]},
    )

    data = np.asarray(D.getData()[0])
    data = np.squeeze(data)

    return data, coord


def plane_extent(horizontal_axis, vertical_axis):
    h = spatial_values[horizontal_axis]
    v = spatial_values[vertical_axis]

    return [
        h[0],
        h[-1],
        v[0],
        v[-1],
    ]


def adaptive_limits(data):
    vmax = finalise_vmax(raw_absmax(data))
    return -vmax, vmax


def fixed_global_limits():
    return -GLOBAL_VMAX, GLOBAL_VMAX


def current_limits(data):
    if COLORBAR_MODE == "global":
        return fixed_global_limits()

    if COLORBAR_MODE == "adaptive":
        return adaptive_limits(data)

    raise ValueError("Unknown COLORBAR_MODE.")

# =========================================================
# GLOBAL MAXIMUM SCAN
# =========================================================

GLOBAL_VMAX = None

if COLORBAR_MODE == "global":

    print("Computing global normalisation...")
    print(f"Field family: {FIELD_FAMILY}")
    print(f"Components: {POLARISATIONS}")
    print(f"Number of timesteps used: {len(timesteps)}")

    vmax_scan = 0.0
    counter = 0
    total = len(POLARISATIONS) * len(timesteps)

    for FIELD in POLARISATIONS:

        for t in timesteps:

            counter += 1

            D = S.Field(
                0,
                FIELD,
                timesteps=t,
            )

            data = D.getData()[0]

            vmax_scan = max(
                vmax_scan,
                raw_absmax(data),
            )

            print(
                f"\rScanning {counter}/{total}: {FIELD}, t={t}",
                end="",
            )

    GLOBAL_VMAX = finalise_vmax(vmax_scan)

    print()
    print(f"GLOBAL_VMAX = {GLOBAL_VMAX}")

else:
    print("Adaptive colourbar mode selected.")
    print("No global scan will be performed.")

# =========================================================
# INITIAL INDICES
# =========================================================

itime0 = 0

ix0 = N_spatial["x"] // 2
iy0 = N_spatial["y"] // 2
iz0 = N_spatial["z"] // 2

t0 = timesteps[itime0]
FIELD0 = DEFAULT_FIELD

# =========================================================
# INITIAL DATA
# =========================================================

field_xy, z_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="z",
    normal_index=iz0,
)

field_xz, y_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="y",
    normal_index=iy0,
)

field_yz, x_coord = load_plane(
    FIELD=FIELD0,
    t=t0,
    normal_axis="x",
    normal_index=ix0,
)

vmin_xy, vmax_xy = current_limits(field_xy)
vmin_xz, vmax_xz = current_limits(field_xz)
vmin_yz, vmax_yz = current_limits(field_yz)

# =========================================================
# FIGURE
# =========================================================

plt.ioff()

fig = plt.figure(
    figsize=(18, 6),
    constrained_layout=True,
)

gs = fig.add_gridspec(
    2,
    3,
    height_ratios=[1.0, 0.06],
)

axs = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]),
]

caxs = [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 2]),
]

fig.suptitle(
    f"{FIELD0} | t index = {itime0}, t = {t0} | "
    f"colourbar mode = {COLORBAR_MODE}",
    fontsize=14,
)

# ---------------------------------------------------------
# xy plane, z fixed
# ---------------------------------------------------------

im_xy = axs[0].imshow(
    field_xy.T,
    extent=plane_extent("x", "y"),
    origin="lower",
    aspect="auto",
    cmap=cmap,
    vmin=vmin_xy,
    vmax=vmax_xy,
)

axs[0].set_title(f"xy plane | z = {z_coord:.4g}")
axs[0].set_xlabel("x")
axs[0].set_ylabel("y")

cbar_xy = fig.colorbar(
    im_xy,
    cax=caxs[0],
    orientation="horizontal",
)
cbar_xy.set_label(FIELD0)

# ---------------------------------------------------------
# xz plane, y fixed
# ---------------------------------------------------------

im_xz = axs[1].imshow(
    field_xz.T,
    extent=plane_extent("x", "z"),
    origin="lower",
    aspect="auto",
    cmap=cmap,
    vmin=vmin_xz,
    vmax=vmax_xz,
)

axs[1].set_title(f"xz plane | y = {y_coord:.4g}")
axs[1].set_xlabel("x")
axs[1].set_ylabel("z")

cbar_xz = fig.colorbar(
    im_xz,
    cax=caxs[1],
    orientation="horizontal",
)
cbar_xz.set_label(FIELD0)

# ---------------------------------------------------------
# yz plane, x fixed
# ---------------------------------------------------------

im_yz = axs[2].imshow(
    field_yz.T,
    extent=plane_extent("y", "z"),
    origin="lower",
    aspect="auto",
    cmap=cmap,
    vmin=vmin_yz,
    vmax=vmax_yz,
)

axs[2].set_title(f"yz plane | x = {x_coord:.4g}")
axs[2].set_xlabel("y")
axs[2].set_ylabel("z")

cbar_yz = fig.colorbar(
    im_yz,
    cax=caxs[2],
    orientation="horizontal",
)
cbar_yz.set_label(FIELD0)

# =========================================================
# SLIDERS
# =========================================================

slider_time = IntSlider(
    min=0,
    max=len(timesteps) - 1,
    step=1,
    value=itime0,
    description="time index",
    continuous_update=False,
)

slider_pol = SelectionSlider(
    options=POLARISATIONS,
    value=FIELD0,
    description="polarisation",
    continuous_update=False,
)

slider_x = IntSlider(
    min=0,
    max=N_spatial["x"] - 1,
    step=1,
    value=ix0,
    description="x index",
    continuous_update=False,
)

slider_y = IntSlider(
    min=0,
    max=N_spatial["y"] - 1,
    step=1,
    value=iy0,
    description="y index",
    continuous_update=False,
)

slider_z = IntSlider(
    min=0,
    max=N_spatial["z"] - 1,
    step=1,
    value=iz0,
    description="z index",
    continuous_update=False,
)

# =========================================================
# UPDATE FUNCTION
# =========================================================

def update_plot(change=None):

    itime = slider_time.value
    FIELD = slider_pol.value

    ix = slider_x.value
    iy = slider_y.value
    iz = slider_z.value

    t = timesteps[itime]

    # -----------------------------------------------------
    # Load current planes
    # -----------------------------------------------------

    field_xy, z_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="z",
        normal_index=iz,
    )

    field_xz, y_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="y",
        normal_index=iy,
    )

    field_yz, x_coord = load_plane(
        FIELD=FIELD,
        t=t,
        normal_axis="x",
        normal_index=ix,
    )

    # -----------------------------------------------------
    # Colour limits
    # -----------------------------------------------------

    vmin_xy, vmax_xy = current_limits(field_xy)
    vmin_xz, vmax_xz = current_limits(field_xz)
    vmin_yz, vmax_yz = current_limits(field_yz)

    # -----------------------------------------------------
    # Update images
    # -----------------------------------------------------

    im_xy.set_data(field_xy.T)
    im_xy.set_clim(vmin_xy, vmax_xy)

    im_xz.set_data(field_xz.T)
    im_xz.set_clim(vmin_xz, vmax_xz)

    im_yz.set_data(field_yz.T)
    im_yz.set_clim(vmin_yz, vmax_yz)

    # -----------------------------------------------------
    # Update titles
    # -----------------------------------------------------

    fig.suptitle(
        f"{FIELD} | t index = {itime}, t = {t} | "
        f"colourbar mode = {COLORBAR_MODE}",
        fontsize=14,
    )

    axs[0].set_title(f"xy plane | z = {z_coord:.4g}")
    axs[1].set_title(f"xz plane | y = {y_coord:.4g}")
    axs[2].set_title(f"yz plane | x = {x_coord:.4g}")

    # -----------------------------------------------------
    # Update colourbars
    # -----------------------------------------------------

    cbar_xy.update_normal(im_xy)
    cbar_xz.update_normal(im_xz)
    cbar_yz.update_normal(im_yz)

    cbar_xy.set_label(FIELD)
    cbar_xz.set_label(FIELD)
    cbar_yz.set_label(FIELD)

    fig.canvas.draw_idle()

# =========================================================
# CONNECT SLIDERS
# =========================================================

for slider in [
    slider_time,
    slider_pol,
    slider_x,
    slider_y,
    slider_z,
]:
    slider.observe(update_plot, names="value")

# =========================================================
# DISPLAY
# =========================================================

controls = VBox(
    [
        HBox([slider_time, slider_pol]),
        HBox([slider_x, slider_y, slider_z]),
    ]
)

display(
    VBox(
        [
            controls,
            fig.canvas,
        ]
    )
)

plt.ion()